# Buscador de Papers - OpenAlex

**OpenAlex** es el catalogo de papers academicos 100% open source mas grande del mundo:
- +250 millones de trabajos academicos de **todas las disciplinas**
- Datos de **instituciones, autores, revistas y conceptos**
- API gratuita sin registro (con email obtiene limite mayor)
- Sucesor oficial de Microsoft Academic Graph

---

## Paso 1: Instalar dependencias

In [ ]:
!pip install requests pandas sumy nltk fpdf2 --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('Listo.')

## Paso 2: Importar librerias y configuracion

In [ ]:
import requests
import pandas as pd
import datetime
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words
from fpdf import FPDF
from google.colab import files
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── OPCIONAL: pon tu email para obtener mayor limite de peticiones ──
# Cambia None por 'tu@email.com' para usar el pool educado (100k req/dia)
MI_EMAIL = None

BASE_URL = 'https://api.openalex.org'

def get_headers():
    if MI_EMAIL:
        return {'User-Agent': f'mailto:{MI_EMAIL}'}
    return {}

print('Librerias importadas.')

## Paso 3: Funciones principales

In [ ]:
def resumir_texto(texto, oraciones=3):
    if not texto or len(texto) < 80:
        return texto or 'Sin abstract disponible.'
    try:
        parser = PlaintextParser.from_string(texto, Tokenizer('english'))
        summarizer = LsaSummarizer(Stemmer('english'))
        summarizer.stop_words = get_stop_words('english')
        return ' '.join(str(s) for s in summarizer(parser.document, oraciones))
    except Exception:
        return texto[:600] + '...' if len(texto) > 600 else texto


def buscar_papers(query, max_results=10, sort='relevance', tipo=None,
                  solo_oa=False, anio_desde=None, anio_hasta=None):
    """Busca en OpenAlex API."""
    sort_map = {
        'relevance':  'relevance_score:desc',
        'citas':      'cited_by_count:desc',
        'reciente':   'publication_date:desc',
        'antiguo':    'publication_date:asc',
    }

    params = {
        'search':        query,
        'per-page':      min(max_results, 200),
        'sort':          sort_map.get(sort, 'relevance_score:desc'),
        'select':        'id,title,authorships,publication_year,publication_date,'
                         'abstract_inverted_index,cited_by_count,open_access,'
                         'primary_location,doi,concepts,type'
    }

    filtros = []
    if solo_oa:
        filtros.append('open_access.is_oa:true')
    if tipo:
        filtros.append(f'type:{tipo}')
    if anio_desde:
        filtros.append(f'publication_year:>{anio_desde - 1}')
    if anio_hasta:
        filtros.append(f'publication_year:<{anio_hasta + 1}')
    if filtros:
        params['filter'] = ','.join(filtros)

    resp = requests.get(f'{BASE_URL}/works', params=params, headers=get_headers(), timeout=20)
    resp.raise_for_status()
    items = resp.json().get('results', [])

    resultados = []
    for p in items:
        # Reconstruir abstract desde indice invertido
        inv_idx = p.get('abstract_inverted_index') or {}
        if inv_idx:
            palabras = {pos: word for word, posiciones in inv_idx.items() for pos in posiciones}
            abstract = ' '.join(palabras[k] for k in sorted(palabras))
        else:
            abstract = ''

        autores = ', '.join(
            a.get('author', {}).get('display_name', '') for a in (p.get('authorships') or [])[:8]
        )
        loc   = p.get('primary_location') or {}
        src   = loc.get('source') or {}
        revista = src.get('display_name', 'N/A')
        oa_url  = (p.get('open_access') or {}).get('oa_url', '')
        conceptos = ', '.join(c.get('display_name','') for c in (p.get('concepts') or [])[:5])

        resultados.append({
            'titulo':    p.get('title', 'Sin titulo'),
            'autores':   autores or 'N/A',
            'anio':      p.get('publication_year', 'N/A'),
            'fecha':     p.get('publication_date', 'N/A'),
            'citas':     p.get('cited_by_count', 0),
            'tipo':      p.get('type', 'N/A'),
            'revista':   revista,
            'conceptos': conceptos,
            'doi':       p.get('doi', 'N/A'),
            'abstract':  abstract,
            'resumen':   resumir_texto(abstract),
            'url':       p.get('id', ''),
            'pdf_url':   oa_url,
        })
    return resultados


def mostrar_html(resultados):
    if not resultados:
        display(HTML('<p style="color:red">No se encontraron resultados.</p>'))
        return
    html = f'<h3>Se encontraron {len(resultados)} trabajos:</h3>'
    for i, p in enumerate(resultados, 1):
        pdf_link = f'<a href="{p["pdf_url"]}" target="_blank">PDF Open Access</a>' if p['pdf_url'] else '<span style="color:#aaa">Sin PDF OA</span>'
        html += f'''
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;
                    background:#f9f9f9;font-family:Arial,sans-serif;">
          <h4 style="margin:0 0 6px;color:#1a0dab;">#{i} — {p['titulo']}</h4>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Autores:</b> {p['autores']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Anio:</b> {p['anio']} &nbsp;|
            <b>Citas:</b> {p['citas']} &nbsp;|
            <b>Tipo:</b> {p['tipo']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Revista/Fuente:</b> {p['revista']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Conceptos:</b> {p['conceptos']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>DOI:</b> {p['doi']}</p>
          <details style="margin-top:8px;">
            <summary style="cursor:pointer;color:#1a0dab;font-size:13px;">Ver abstract completo</summary>
            <p style="font-size:12px;color:#333;margin-top:6px;">{p['abstract'] or 'No disponible.'}</p>
          </details>
          <div style="background:#eef6ff;border-left:4px solid #1a73e8;
                      padding:10px;margin-top:10px;border-radius:4px;">
            <b style="font-size:13px;">Resumen automatico:</b>
            <p style="font-size:13px;margin:4px 0;">{p['resumen']}</p>
          </div>
          <p style="margin-top:8px;font-size:13px;">
            <a href="{p['url']}" target="_blank">Ver en OpenAlex</a>
            &nbsp;|&nbsp; {pdf_link}
          </p>
        </div>'''
    display(HTML(html))


def exportar_csv(resultados, nombre=None):
    if not resultados: print('Sin resultados.'); return
    nombre = nombre or f'openalex_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    pd.DataFrame(resultados).to_csv(nombre, index=False, encoding='utf-8-sig')
    print(f'Archivo: {nombre}')
    files.download(nombre)


def exportar_pdf(resultados, nombre=None):
    if not resultados: print('Sin resultados.'); return
    nombre = nombre or f'openalex_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.pdf'
    safe = lambda t: (str(t) or '').encode('latin-1', 'replace').decode('latin-1')
    pdf = FPDF()
    pdf.set_auto_page_break(True, 15)
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 16)
    pdf.cell(0, 10, 'Resultados - OpenAlex', ln=True, align='C')
    pdf.set_font('Helvetica', '', 10)
    pdf.cell(0, 8, f'Generado: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}', ln=True, align='C')
    pdf.ln(6)
    for i, p in enumerate(resultados, 1):
        pdf.set_font('Helvetica', 'B', 12)
        pdf.multi_cell(0, 7, safe(f'{i}. {p["titulo"]}'))
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe(f'Autores: {p["autores"]}'))
        pdf.multi_cell(0, 6, safe(f'Anio: {p["anio"]}  |  Citas: {p["citas"]}  |  Tipo: {p["tipo"]}'))
        pdf.multi_cell(0, 6, safe(f'Revista: {p["revista"]}  |  Conceptos: {p["conceptos"]}'))
        pdf.multi_cell(0, 6, safe(f'DOI: {p["doi"]}'))
        pdf.ln(2)
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Resumen automatico:', ln=True)
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe(p['resumen']))
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Abstract:', ln=True)
        pdf.set_font('Helvetica', '', 8)
        pdf.multi_cell(0, 5, safe(p['abstract']))
        pdf.ln(4)
        pdf.set_draw_color(180, 180, 180)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(4)
    pdf.output(nombre)
    print(f'Archivo: {nombre}')
    files.download(nombre)


print('Funciones cargadas.')

## Paso 4: Interfaz interactiva

In [ ]:
TIPOS = [
    ('Todos los tipos', ''),
    ('Articulo (article)', 'article'),
    ('Revision (review)', 'review'),
    ('Libro (book)', 'book'),
    ('Capitulo de libro (book-chapter)', 'book-chapter'),
    ('Tesis (dissertation)', 'dissertation'),
    ('Dataset (dataset)', 'dataset'),
]

w_query   = widgets.Text(value='', placeholder='Ej: solar energy machine learning, protein folding...',
                         description='Busqueda:', layout=widgets.Layout(width='70%'),
                         style={'description_width': '80px'})
w_tipo    = widgets.Dropdown(options=TIPOS, value='', description='Tipo:',
                              layout=widgets.Layout(width='48%'), style={'description_width': '80px'})
w_sort    = widgets.Dropdown(
    options=[('Relevancia','relevance'),('Mas citas','citas'),('Mas reciente','reciente'),('Mas antiguo','antiguo')],
    value='relevance', description='Ordenar:', layout=widgets.Layout(width='48%'),
    style={'description_width': '80px'})
w_max     = widgets.IntSlider(value=10, min=1, max=200, step=5, description='Cantidad:',
                               layout=widgets.Layout(width='60%'), style={'description_width': '80px'})
w_oa      = widgets.Checkbox(value=False, description='Solo Open Access (PDF gratuito)', indent=False)
w_desde   = widgets.IntText(value=2015, description='Desde:', style={'description_width': '60px'},
                             layout=widgets.Layout(width='150px'))
w_hasta   = widgets.IntText(value=2024, description='Hasta:', style={'description_width': '60px'},
                             layout=widgets.Layout(width='150px'))
w_filtrar = widgets.Checkbox(value=False, description='Filtrar por rango de anios', indent=False)

btn_buscar = widgets.Button(description='Buscar', button_style='primary', icon='search',
                             layout=widgets.Layout(width='130px', height='36px'))
btn_csv    = widgets.Button(description='CSV', button_style='success', icon='download',
                             layout=widgets.Layout(width='110px', height='36px'), disabled=True)
btn_pdf    = widgets.Button(description='PDF', button_style='warning', icon='file',
                             layout=widgets.Layout(width='110px', height='36px'), disabled=True)

out_status  = widgets.Output()
out_results = widgets.Output()
resultados_globales = []

def on_buscar(b):
    global resultados_globales
    query = w_query.value.strip()
    if not query:
        with out_status: clear_output(); print('Escribe un termino de busqueda.')
        return
    btn_buscar.disabled = True; btn_csv.disabled = True; btn_pdf.disabled = True
    with out_status: clear_output(); print(f'Buscando "{query}" en OpenAlex...')
    with out_results: clear_output()
    try:
        desde = w_desde.value if w_filtrar.value else None
        hasta = w_hasta.value if w_filtrar.value else None
        resultados_globales = buscar_papers(
            query, w_max.value, w_sort.value,
            w_tipo.value or None, w_oa.value, desde, hasta
        )
        with out_status: clear_output(); print(f'{len(resultados_globales)} trabajos encontrados.')
        with out_results: mostrar_html(resultados_globales)
        btn_csv.disabled = False; btn_pdf.disabled = False
    except Exception as e:
        with out_status: clear_output(); print(f'Error: {e}')
    finally:
        btn_buscar.disabled = False

btn_buscar.on_click(on_buscar)
btn_csv.on_click(lambda b: exportar_csv(resultados_globales))
btn_pdf.on_click(lambda b: exportar_pdf(resultados_globales))

display(HTML('<h3 style="font-family:Arial">Buscador de Papers - OpenAlex</h3>'))
display(widgets.VBox([
    w_query,
    widgets.HBox([w_tipo, w_sort]),
    widgets.HBox([w_max]),
    widgets.HBox([w_oa]),
    widgets.HBox([w_filtrar, w_desde, w_hasta]),
    widgets.HBox([btn_buscar, btn_csv, btn_pdf]),
    out_status, out_results
]))

## Paso 5 (opcional): Busqueda por codigo directo

In [ ]:
QUERY      = 'machine learning materials science'  # Busqueda
MAX        = 5                                     # Cantidad (max 200)
SORT       = 'citas'                               # 'relevance','citas','reciente','antiguo'
TIPO       = 'article'                             # '' para todos, 'article','review','book'...
SOLO_OA    = True                                  # True = solo open access
ANIO_DESDE = 2019                                  # None para sin filtro
ANIO_HASTA = 2024                                  # None para sin filtro

resultados = buscar_papers(QUERY, MAX, SORT, TIPO, SOLO_OA, ANIO_DESDE, ANIO_HASTA)
mostrar_html(resultados)

# exportar_csv(resultados)
# exportar_pdf(resultados)

---

## Licencias, Politicas de Datos y Uso de la API

### Quien opera OpenAlex
OpenAlex es desarrollado por **OurResearch**, una organizacion sin fines de lucro dedicada a hacer la investigacion cientifica mas abierta y accesible. Es el sucesor oficial de Microsoft Academic Graph (MAG), que fue descontinuado en 2021. Cuenta con el apoyo de la Andrew W. Mellon Foundation.

---

### Licencia de los datos

| Tipo de dato | Licencia |
|---|---|
| **Todos los metadatos de OpenAlex** (titulos, autores, abstracts, citas, conceptos, instituciones) | **CC0 1.0 — Dominio Publico Universal** |
| API (uso en tiempo real) | **Completamente gratuita, sin restricciones de uso** |
| Snapshots del dataset completo | **CC0 1.0** — disponibles para descarga masiva |

**CC0 (Creative Commons Zero)** significa que OurResearch renuncia a todos los derechos de autor sobre los datos. Puedes usar, copiar, modificar y distribuir los datos **para cualquier proposito, incluyendo comercial, sin necesidad de pedir permiso ni dar atribucion** (aunque la atribucion es bienvenida).

> OpenAlex es la unica de las cuatro fuentes de este proyecto con licencia CC0 total sobre sus metadatos. Es la opcion mas permisiva para investigacion y desarrollo.

---

### Terminos de uso de la API

- **Completamente gratuita** sin registro
- **Sin API key (anonimo):** 10 peticiones por segundo, 100,000 peticiones por dia
- **Con email en el header (polite pool):** mismo limite pero acceso prioritario y soporte
- **Sin restricciones de uso comercial** en los datos de la API
- **Snapshots para descarga masiva:** disponibles via Amazon S3 (datos actualizados mensualmente)
- **Prohibido:** saturar intencionalmente los servidores con peticiones abusivas
- Documentacion oficial: https://docs.openalex.org

**Como activar el polite pool (recomendado):**
En el Paso 2 de este notebook, cambia `MI_EMAIL = None` por tu email institucional:
```python
MI_EMAIL = 'tu.nombre@universidad.edu.co'
```

---

### Privacidad y seguridad de datos

- Este notebook **no recopila ni almacena** datos personales
- Si proporcionas tu email en `MI_EMAIL`, este se envia en el header HTTP de cada peticion a OpenAlex (no a terceros)
- OpenAlex puede registrar IPs y emails del polite pool para gestion de uso
- OurResearch tiene una politica de privacidad publica en: https://ourresearch.org/privacy
- Los archivos CSV y PDF generados se descargan localmente; **no se transmiten a servidores externos**

---

### Cobertura y calidad de los datos

- **+250 millones** de trabajos academicos de todas las disciplinas y paises
- Datos actualizados **diariamente**
- Informacion de autores, instituciones, revistas, conceptos y financiadores
- Abstracts disponibles para aproximadamente el 57% de los trabajos (limitacion de derechos de editoriales)
- Los abstracts se almacenan como **indice invertido** por restricciones legales con editoriales; este notebook los reconstruye automaticamente

---

### Nota de citacion academica

Para citar OpenAlex como fuente de datos en publicaciones:
> Priem, J., Piwowar, H., & Orr, R. (2022). OpenAlex: A fully-open index of the world's research literature. *arXiv*. https://doi.org/10.48550/arXiv.2205.01833